In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, privacy

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat = cfg["catalog"]
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print("PASS" if ok else "FAIL", name, detail)

def one(sql):
    return spark.sql(sql).first()[0]

In [0]:
rows = privacy.status(spark, cfg, env)
missing = [(r["layer"], r["table"], r["column"]) for r in rows if r["actual"] != r["wanted"]]
planned = {(r["layer"], r["table"], r["column"]) for r in rows}
layers = privacy.load()["layers"]
extra = [k for k in privacy.masks_now(spark, cat) if k[0] in layers and k not in planned]
count = {}
for r in rows:
    count[r["layer"]] = count.get(r["layer"], 0) + 1
print("INFO masked columns per layer:", count)

check("columns to mask were found", len(rows) > 0, len(rows))
check("every listed column has its mask", not missing, str(missing[:5]))
check("no mask outside the config", not extra, str(extra[:5]))
check("raw rows in quarantine are covered", ("silver", "quarantine", "raw_row") in planned)

person = "patients" if cfg["code"] == "bhg" else "customers"
mine = [g for g in privacy.load()["clear"][env] if one(f"SELECT is_account_group_member('{g}')")]
real = one(f"SELECT count(*) FROM {cat}.silver.{person} WHERE aadhaar RLIKE '^[0-9]+$'")
check("you read real values only if your group is clear", (real > 0) == bool(mine), f"groups {mine}, readable ids {real}")

print(f"\nt15_masks_applied: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t15_masks_applied failed")